In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle

In [4]:
def interpolate_column(column, fillna_method='linear'):
    """Interpolates a Pandas Series column, handling vector elements."""
    if isinstance(column.iloc[0], list):
        # Flatten the lists into a single list of values for interpolation
        valid_data = column.dropna() # Drop rows with NaN in the column
        if not valid_data.empty: # Proceed only if there's valid data to flatten
            flat_list = [item for sublist in valid_data.tolist() for item in sublist]
            # Create a Series from the flattened list, interpolate, and reshape
            interpolated_series = pd.Series(flat_list).interpolate(method=fillna_method, limit_direction='both')
            array_length = len(valid_data.iloc[0]) # Get array length from the first valid entry
            interpolated_lists = [interpolated_series[i:i + array_length].tolist() for i in range(0, len(interpolated_series), array_length)]

            # Reconstruct the interpolated column, aligning with original indices
            interpolated_column = pd.Series(index=column.index, dtype='object') # Use 'object' dtype for lists
            valid_indices = valid_data.index # Indices where original data was valid
            for i, idx in enumerate(valid_indices):
                interpolated_column[idx] = interpolated_lists[i]

            # Fill remaining NaN indices with interpolation (important for resampled data)
            return interpolated_column.interpolate(method=fillna_method, limit_direction='both')
        else:
            return column # Return original column if no valid data to interpolate
    else:
        # Interpolate as usual for scalar values
        return column.interpolate(method=fillna_method, limit_direction='both')

In [5]:
data = pickle.load(open('test.pkl','rb'))

In [6]:
df = pd.DataFrame(data)
df.columns

Index(['BDS_BTS:PSC2_D5496:I_CSET', 'BDS_BTS:PSQ_D5552:I_CSET',
       'BDS_BTS:PSQ_D5501:I_RD', 'BDS_BTS:PSQ_D5552:I_RD',
       'BDS_BTS:BPM_D5513:YPOS_RD', 'BDS_BTS:BPM_D5565:TISMAG161_2_RD',
       'BDS_BTS:BPM_D5565:TISRAW4_RD', 'BDS_BTS:BPM_D5513:TISMAG161_2_RD',
       'BDS_BTS:PSQ_D5509:I_RD', 'BDS_BTS:BPM_D5513:TISRAW3_RD',
       'BDS_BTS:PSC1_D5563:I_RD', 'BDS_BTS:BPM_D5513:TISMAG161_1_RD',
       'BDS_BTS:BPM_D5513:CURRENT_RD', 'BDS_BTS:BPM_D5565:TISMAG161_4_RD',
       'BDS_BTS:BPM_D5565:TISMAG161_1_RD', 'BDS_BTS:BPM_D5565:CURRENT_RD',
       'BDS_BTS:BPM_D5513:TISRAW2_RD', 'BDS_BTS:BPM_D5513:PHASE_RD',
       'BDS_BTS:PSC1_D5563:I_CSET', 'BDS_BTS:PSQ_D5509:I_CSET',
       'BDS_BTS:BPM_D5513:MAG_RD', 'BDS_BTS:BPM_D5513:TISRAW4_RD',
       'BDS_BTS:PSC1_D5496:I_RD', 'BDS_BTS:PSC2_D5563:I_CSET',
       'BDS_BTS:BPM_D5565:YPOS_RD', 'BDS_BTS:PSQ_D5559:I_CSET',
       'BDS_BTS:BPM_D5565:TISMAG161_3_RD', 'BDS_BTS:PSC2_D5496:I_RD',
       'BDS_BTS:BPM_D5565:TISRAW3_RD', 'BDS_BTS:

In [8]:
df

,BDS_BTS:PSC2_D5496:I_CSET,BDS_BTS:PSQ_D5552:I_CSET,BDS_BTS:PSQ_D5501:I_RD,BDS_BTS:PSQ_D5552:I_RD,BDS_BTS:BPM_D5513:YPOS_RD,BDS_BTS:BPM_D5565:TISMAG161_2_RD,BDS_BTS:BPM_D5565:TISRAW4_RD,BDS_BTS:BPM_D5513:TISMAG161_2_RD,BDS_BTS:PSQ_D5509:I_RD,BDS_BTS:BPM_D5513:TISRAW3_RD,...,BDS_BTS:PSC2_D5563:I_RD,BDS_BTS:BPM_D5565:MAG_RD,BDS_BTS:BPM_D5565:XPOS_RD,BDS_BTS:BPM_D5513:XPOS_RD,BDS_BTS:PSC1_D5496:I_CSET,BDS_BTS:PSQ_D5501:I_CSET,BDS_BTS:BPM_D5513:TISRAW1_RD,BDS_BTS:BPM_D5565:TISRAW1_RD,BDS_BTS:BPM_D5513:TISMAG161_4_RD,BDS_BTS:BPM_D5565:TISRAW2_RD
2025-02-19 03:31:39.800,0.0,113.825,96.760,113.736,-0.200405,47.697038,"[2353.3823529411766, 1561.5, 444.3823529411764...",56.274298,84.303,"[-1870.735294117647, -1735.7058823529412, -128...",...,0.0,0.003281,0.183723,0.395098,-13.43,96.859,"[-1776.8529411764707, -1589.0294117647059, -10...","[1993.7058823529412, 1370.3235294117646, 459.7...",42.895729,"[2212.9117647058824, 1474.3235294117646, 414.9..."
2025-02-19 03:31:40.000,0.0,113.825,96.760,113.744,-0.195964,47.697038,"[2353.3823529411766, 1561.5, 444.3823529411764...",56.274298,84.303,"[-1870.735294117647, -1735.7058823529412, -128...",...,0.0,0.003366,0.131196,0.349518,-13.43,96.859,"[-1776.8529411764707, -1589.0294117647059, -10...","[1993.7058823529412, 1370.3235294117646, 459.7...",42.895729,"[2212.9117647058824, 1474.3235294117646, 414.9..."
2025-02-19 03:31:40.200,0.0,113.825,96.760,113.744,-0.175982,47.697038,"[2353.3823529411766, 1561.5, 444.3823529411764...",56.274298,84.303,"[-1870.735294117647, -1735.7058823529412, -128...",...,0.0,0.003242,0.039138,0.244457,-13.43,96.859,"[-1776.8529411764707, -1589.0294117647059, -10...","[1993.7058823529412, 1370.3235294117646, 459.7...",42.895729,"[2212.9117647058824, 1474.3235294117646, 414.9..."
2025-02-19 03:31:40.400,0.0,113.825,96.760,113.744,-0.180427,47.185707,"[2306.9411764705883, 1531.235294117647, 460.94...",55.734934,84.303,"[-1756.3823529411766, -1646.4411764705883, -12...",...,0.0,0.003219,0.060687,0.270498,-13.43,96.859,"[-1760.0882352941178, -1540.7941176470588, -10...","[1953.8529411764705, 1351.3235294117646, 471.8...",41.655390,"[2186.470588235294, 1457.6470588235293, 429.47..."
2025-02-19 03:31:40.600,0.0,113.825,96.760,113.744,-0.202801,47.185707,"[2306.9411764705883, 1531.235294117647, 460.94...",55.734934,84.303,"[-1756.3823529411766, -1646.4411764705883, -12...",...,0.0,0.003258,0.178259,0.404787,-13.43,96.859,"[-1760.0882352941178, -1540.7941176470588, -10...","[1953.8529411764705, 1351.3235294117646, 471.8...",41.655390,"[2186.470588235294, 1457.6470588235293, 429.47..."
2025-02-19 03:31:40.800,0.0,113.825,96.768,113.744,-0.206508,47.185707,"[2306.9411764705883, 1531.235294117647, 460.94...",55.734934,84.303,"[-1756.3823529411766, -1646.4411764705883, -12...",...,0.0,0.003216,0.207961,0.448050,-13.43,96.859,"[-1760.0882352941178, -1540.7941176470588, -10...","[1953.8529411764705, 1351.3235294117646, 471.8...",41.655390,"[2186.470588235294, 1457.6470588235293, 429.47..."
2025-02-19 03:31:41.000,0.0,113.825,96.768,113.744,-0.207639,47.185707,"[2306.9411764705883, 1531.235294117647, 460.94...",55.734934,84.303,"[-1756.3823529411766, -1646.4411764705883, -12...",...,0.0,0.003158,0.235907,0.480900,-13.43,96.859,"[-1760.0882352941178, -1540.7941176470588, -10...","[1953.8529411764705, 1351.3235294117646, 471.8...",41.655390,"[2186.470588235294, 1457.6470588235293, 429.47..."
2025-02-19 03:31:41.200,0.0,113.825,96.768,113.744,-0.188632,47.185707,"[2306.9411764705883, 1531.235294117647, 460.94...",55.734934,84.303,"[-1756.3823529411766, -1646.4411764705883, -12...",...,0.0,0.003337,0.194482,0.436069,-13.43,96.859,"[-1760.0882352941178, -1540.7941176470588, -10...","[1953.8529411764705, 1351.3235294117646, 471.8...",41.655390,"[2186.470588235294, 1457.6470588235293, 429.47..."
2025-02-19 03:31:41.400,0.0,113.825,96.768,113.744,-0.197264,48.119058,"[2426.6764705882356, 1660.5882352941178, 578.6...",56.757064,84.303,"[-1836.764705882353, -1741.4117647058822, -131...",...,0

In [34]:
PVs = ['BDS_BTS:PSQ_D5501:I_CSET','BDS_BTS:BPM_D5513:TISRAW1_RD']
df = df[PVs].iloc[:8,:]

In [35]:
df.to_dict()

{'BDS_BTS:PSQ_D5501:I_CSET': {Timestamp('2025-02-19 03:31:39.800000'): 96.859,
  Timestamp('2025-02-19 03:31:40'): 96.859,
  Timestamp('2025-02-19 03:31:40.200000'): 96.859,
  Timestamp('2025-02-19 03:31:40.400000'): 96.859,
  Timestamp('2025-02-19 03:31:40.600000'): 96.859,
  Timestamp('2025-02-19 03:31:40.800000'): 96.859,
  Timestamp('2025-02-19 03:31:41'): 96.859,
  Timestamp('2025-02-19 03:31:41.200000'): 96.859},
 'BDS_BTS:BPM_D5513:TISRAW1_RD': {Timestamp('2025-02-19 03:31:39.800000'): array([-1776.85294118, -1589.02941176, -1085.85294118,  -297.02941176,
           685.14705882,  1640.97058824,  2396.14705882,  2823.97058824,
          2784.14705882,  2242.97058824,  1293.14705882,    36.97058824,
         -1337.85294118, -2584.02941176, -3439.85294118, -3841.02941176,
         -3667.85294118, -3040.02941176, -1996.85294118,  -713.02941176,
           546.14705882,  1649.97058824,  2495.14705882,  2999.97058824,
          3122.14705882,  2878.97058824,  2403.14705882,  1779.970

In [36]:
array([array([np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]),
       array([-1776.85294118, -1589.02941176, -1085.85294118,  -297.02941176,
                685.14705882,  1640.97058824,  2396.14705882,  2823.97058824,
               2784.14705882,  2242.97058824,  1293.14705882,    36.97058824,
              -1337.85294118, -2584.02941176, -3439.85294118, -3841.02941176,
              -3667.85294118, -3040.02941176, -1996.85294118,  -713.02941176,
                546.14705882,  1649.97058824,  2495.14705882,  2999.97058824,
               3122.14705882,  2878.97058824,  2403.14705882,  1779.97058824,
               1127.14705882,   502.97058824,   -10.85294118,  -390.02941176,
               -660.85294118,  -778.02941176,  -859.85294118,  -914.02941176,
               -938.85294118,  -959.02941176,  -958.85294118,  -951.02941176,
               -948.85294118,  -916.02941176,  -841.85294118,  -714.02941176,
               -569.85294118,  -391.02941176,  -205.85294118,    25.97058824,
                244.14705882,   422.97058824,   594.14705882,   697.97058824,
                784.14705882,   829.97058824,   801.14705882,   789.97058824,
                735.14705882,   662.97058824,   597.14705882,   535.97058824,
                435.14705882,   307.97058824,   140.14705882,  -140.02941176,
               -489.85294118,  -949.02941176, -1392.85294118, -1664.02941176]),
       array([-1776.85294118, -1589.02941176, -1085.85294118,  -297.02941176,
                685.14705882,  1640.97058824,  2396.14705882,  2823.97058824,
               2784.14705882,  2242.97058824,  1293.14705882,    36.97058824,
              -1337.85294118, -2584.02941176, -3439.85294118, -3841.02941176,
              -3667.85294118, -3040.02941176, -1996.85294118,  -713.02941176,
                546.14705882,  1649.97058824,  2495.14705882,  2999.97058824,
               3122.14705882,  2878.97058824,  2403.14705882,  1779.97058824,
               1127.14705882,   502.97058824,   -10.85294118,  -390.02941176,
               -660.85294118,  -778.02941176,  -859.85294118,  -914.02941176,
               -938.85294118,  -959.02941176,  -958.85294118,  -951.02941176,
               -948.85294118,  -916.02941176,  -841.85294118,  -714.02941176,
               -569.85294118,  -391.02941176,  -205.85294118,    25.97058824,
                244.14705882,   422.97058824,   594.14705882,   697.97058824,
                784.14705882,   829.97058824,   801.14705882,   789.97058824,
                735.14705882,   662.97058824,   597.14705882,   535.97058824,
                435.14705882,   307.97058824,   140.14705882,  -140.02941176,
               -489.85294118,  -949.02941176, -1392.85294118, -1664.02941176]),
       array([-1760.08823529, -1540.79411765, -1070.08823529,  -277.79411765,
                653.91176471,  1586.20588235,  2328.91176471,  2760.20588235,
               2732.91176471,  2211.20588235,  1296.91176471,    41.20588235,
              -1291.08823529, -2529.79411765, -3361.08823529, -3733.79411765,
              -3603.08823529, -3020.79411765, -1998.08823529,  -727.79411765,
                522.91176471,  1607.20588235,  2436.91176471,  2936.20588235,
               3064.91176471,  2825.20588235,  2380.91176471,  1750.20588235,
               1101.91176471,   515.20588235,   -15.08823529,  -391.79411765,
               -651.08823529,  -765.79411765,  -868.08823529,  -869.79411765,
               -926.08823529,  -939.79411765,  -939.08823529,  -933.79411765,
               -942.08823529,  -897.79411765,  -829.08823529,  -726.79411765,
               -560.08823529,  -393.79411765,  -180.08823529,    23.20588235,
                231.91176471,   409.20588235,   592.91176471,   704.20588235,
                767.91176471,   818.20588235,   805.91176471,   787.20588235,
                716.91176471,   640.20588235,   601.91176471,   512.20588235,
                425.91176471,   318.20588235,   153.91176471,  -135.79411765,
               -479.08823529,  -914.79411765, -1344.08823529, -1644.79411765]),
       array([-1760.08823529, -1540.79411765, -1070.08823529,  -277.79411765,
                653.91176471,  1586.20588235,  2328.91176471,  2760.20588235,
               2732.91176471,  2211.20588235,  1296.91176471,    41.20588235,
              -1291.08823529, -2529.79411765, -3361.08823529, -3733.79411765,
              -3603.08823529, -3020.79411765, -1998.08823529,  -727.79411765,
                522.91176471,  1607.20588235,  2436.91176471,  2936.20588235,
               3064.91176471,  2825.20588235,  2380.91176471,  1750.20588235,
               1101.91176471,   515.20588235,   -15.08823529,  -391.79411765,
               -651.08823529,  -765.79411765,  -868.08823529,  -869.79411765,
               -926.08823529,  -939.79411765,  -939.08823529,  -933.79411765,
               -942.08823529,  -897.79411765,  -829.08823529,  -726.79411765,
               -560.08823529,  -393.79411765,  -180.08823529,    23.20588235,
                231.91176471,   409.20588235,   592.91176471,   704.20588235,
                767.91176471,   818.20588235,   805.91176471,   787.20588235,
                716.91176471,   640.20588235,   601.91176471,   512.20588235,
                425.91176471,   318.20588235,   153.91176471,  -135.79411765,
               -479.08823529,  -914.79411765, -1344.08823529, -1644.79411765]),
       array([-1760.08823529, -1540.79411765, -1070.08823529,  -277.79411765,
                653.91176471,  1586.20588235,  2328.91176471,  2760.20588235,
               2732.91176471,  2211.20588235,  1296.91176471,    41.20588235,
              -1291.08823529, -2529.79411765, -3361.08823529, -3733.79411765,
              -3603.08823529, -3020.79411765, -1998.08823529,  -727.79411765,
                522.91176471,  1607.20588235,  2436.91176471,  2936.20588235,
               3064.91176471,  2825.20588235,  2380.91176471,  1750.20588235,
               1101.91176471,   515.20588235,   -15.08823529,  -391.79411765,
               -651.08823529,  -765.79411765,  -868.08823529,  -869.79411765,
               -926.08823529,  -939.79411765,  -939.08823529,  -933.79411765,
               -942.08823529,  -897.79411765,  -829.08823529,  -726.79411765,
               -560.08823529,  -393.79411765,  -180.08823529,    23.20588235,
                231.91176471,   409.20588235,   592.91176471,   704.20588235,
                767.91176471,   818.20588235,   805.91176471,   787.20588235,
                716.91176471,   640.20588235,   601.91176471,   512.20588235,
                425.91176471,   318.20588235,   153.91176471,  -135.79411765,
               -479.08823529,  -914.79411765, -1344.08823529, -1644.79411765]),
       array([-1760.08823529, -1540.79411765, -1070.08823529,  -277.79411765,
                653.91176471,  1586.20588235,  2328.91176471,  2760.20588235,
               2732.91176471,  2211.20588235,  1296.91176471,    41.20588235,
              -1291.08823529, -2529.79411765, -3361.08823529, -3733.79411765,
              -3603.08823529, -3020.79411765, -1998.08823529,  -727.79411765,
                522.91176471,  1607.20588235,  2436.91176471,  2936.20588235,
               3064.91176471,  2825.20588235,  2380.91176471,  1750.20588235,
               1101.91176471,   515.20588235,   -15.08823529,  -391.79411765,
               -651.08823529,  -765.79411765,  -868.08823529,  -869.79411765,
               -926.08823529,  -939.79411765,  -939.08823529,  -933.79411765,
               -942.08823529,  -897.79411765,  -829.08823529,  -726.79411765,
               -560.08823529,  -393.79411765,  -180.08823529,    23.20588235,
                231.91176471,   409.20588235,   592.91176471,   704.20588235,
                767.91176471,   818.20588235,   805.91176471,   787.20588235,
                716.91176471,   640.20588235,   601.91176471,   512.20588235,
                425.91176471,   318.20588235,   153.91176471,  -135.79411765,
               -479.08823529,  -914.79411765, -1344.08823529, -1644.79411765]),
       array([-1760.08823529, -1540.79411765, -1070.08823529,  -277.79411765,
                653.91176471,  1586.20588235,  2328.91176471,  2760.20588235,
               2732.91176471,  2211.20588235,  1296.91176471,    41.20588235,
              -1291.08823529, -2529.79411765, -3361.08823529, -3733.79411765,
              -3603.08823529, -3020.79411765, -1998.08823529,  -727.79411765,
                522.91176471,  1607.20588235,  2436.91176471,  2936.20588235,
               3064.91176471,  2825.20588235,  2380.91176471,  1750.20588235,
               1101.91176471,   515.20588235,   -15.08823529,  -391.79411765,
               -651.08823529,  -765.79411765,  -868.08823529,  -869.79411765,
               -926.08823529,  -939.79411765,  -939.08823529,  -933.79411765,
               -942.08823529,  -897.79411765,  -829.08823529,  -726.79411765,
               -560.08823529,  -393.79411765,  -180.08823529,    23.20588235,
                231.91176471,   409.20588235,   592.91176471,   704.20588235,
                767.91176471,   818.20588235,   805.91176471,   787.20588235,
                716.91176471,   640.20588235,   601.91176471,   512.20588235,
                425.91176471,   318.20588235,   153.91176471,  -135.79411765,
               -479.08823529,  -914.79411765, -1344.08823529, -1644.79411765])],
      dtype=object)

array([array([-1776.85294118, -1589.02941176, -1085.85294118,  -297.02941176,
                685.14705882,  1640.97058824,  2396.14705882,  2823.97058824,
               2784.14705882,  2242.97058824,  1293.14705882,    36.97058824,
              -1337.85294118, -2584.02941176, -3439.85294118, -3841.02941176,
              -3667.85294118, -3040.02941176, -1996.85294118,  -713.02941176,
                546.14705882,  1649.97058824,  2495.14705882,  2999.97058824,
               3122.14705882,  2878.97058824,  2403.14705882,  1779.97058824,
               1127.14705882,   502.97058824,   -10.85294118,  -390.02941176,
               -660.85294118,  -778.02941176,  -859.85294118,  -914.02941176,
               -938.85294118,  -959.02941176,  -958.85294118,  -951.02941176,
               -948.85294118,  -916.02941176,  -841.85294118,  -714.02941176,
               -569.85294118,  -391.02941176,  -205.85294118,    25.97058824,
                244.14705882,   422.97058824,   594.14705882,   

In [17]:
df['BDS_BTS:BPM_D5513:TISRAW1_RD'] = df['BDS_BTS:BPM_D5513:TISRAW1_RD'].apply(
    lambda x: np.array(x, dtype=np.float32)
)

# Verify the dtype of the arrays in the column
print(df['BDS_BTS:BPM_D5513:TISRAW1_RD'].iloc[0].dtype)  # Sh

float32


In [24]:
arr = np.stack(df['BDS_BTS:BPM_D5513:TISRAW1_RD'].values,dtype=np.float32)
arr.shape

(8, 68)

In [ ]:
# Helper function to check if a vector is valid: it must be a NumPy array of length 68 without any NaN values.
def is_valid_vector(x,length=68):
    return isinstance(x, np.ndarray) and (len(x) == length) and (not np.isnan(x).any())

def valid_df_rows(df, vector_cols, lengths, scalar_cols=None):

    vactor_masks = np.array([True]*len(df))
    for cols, l in zip(vector_cols, lengths):
        vactor_masks = vactor_masks & df[cols].applymap(is_valid_vector,l).all(axis=1)
    if scalar_cols is None:
        scalar_cols = set(df.columns) - [set(col) for col in vector_cols]
    scalar_mask = df[scalar_cols].notna().all(axis=1)

    return df[vactor_masks & scalar_mask]


mask = vector_mask & scalar_mask
df_filtered = 



In [38]:
import numpy as np

def is_valid_vector(x, length=68):
    return isinstance(x, np.ndarray) and (len(x) == length) and (not np.isnan(x).any())

def valid_df_rows(df, vector_cols, lengths, scalar_cols=None):
    # Start with a mask of all True values.
    vector_mask = np.array([True] * len(df))
    
    # For each vector column and its expected length, update the mask.
    for col, l in zip(vector_cols, lengths):
        # For each element in the column, check if it's a valid vector.
        vector_mask &= df[col].apply(lambda x: is_valid_vector(x, length=l))
    
    # If scalar_cols is not provided, consider all columns not in vector_cols.
    if scalar_cols is None:
        scalar_cols = list(set(df.columns) - set(vector_cols))
    
    # Create a mask that checks that all scalar columns contain no NaN.
    scalar_mask = df[scalar_cols].notna().all(axis=1)
    
    # Return only the rows where both masks are True.
    return df[vector_mask & scalar_mask]


In [50]:
import numpy as np
import pandas as pd

def is_valid_vector(x, length=68):
    """Check if x is a NumPy array of the given length with no NaNs."""
    return isinstance(x, np.ndarray) and (len(x) == length) and (not np.isnan(x).any())

def valid_df_rows_iter(df, vector_cols, lengths, scalar_cols=None):
    """
    Filter rows by iterating over each row once.
    vector_cols: list of lists of columns. Each sublist is a group where all columns must be valid vectors.
    lengths: list of expected lengths for each group.
    scalar_cols: list of scalar column names; if None, all columns not in any vector group.
    """
    # Determine scalar_cols only once.
    if scalar_cols is None:
        all_vector_cols = {col for group in vector_cols for col in group}
        scalar_cols = list(set(df.columns) - all_vector_cols)
    
    valid_indices = []
    for idx, row in df.iterrows():
        valid = True
        # Check each vector group.
        for group, l in zip(vector_cols, lengths):
            for col in group:
                x = row[col]
                # Inline the check:
                if not (isinstance(x, np.ndarray) and (x.shape[0] == l) and (not np.isnan(x).any())):
                    valid = False
                    break
            if not valid:
                break
        
        # Check scalar columns.
        if valid and not pd.notna(row[scalar_cols]).all():
            valid = False
        
        if valid:
            valid_indices.append(idx)
    
    return df.loc[valid_indices]

# ----------------------
# Test example:
# ----------------------

# Create sample vectors.
valid_vector4 = np.arange(4, dtype=np.float32)   # Valid vector of length 4.
valid_vector5 = np.arange(5, dtype=np.float32)   # Valid vector of length 5.
invalid_vector_short = np.arange(2, dtype=np.float32)  # Invalid: too short.
invalid_vector_nan = np.arange(4, dtype=np.float32)    # Invalid: contains a NaN.
invalid_vector_nan[2] = np.nan

# Construct a test DataFrame with three vector columns and two scalar columns.
data = {
    "vec1": [valid_vector4, valid_vector4, invalid_vector_short, valid_vector4, invalid_vector_nan],
    "vec2": [valid_vector4, invalid_vector_nan, valid_vector4, valid_vector4, valid_vector4],
    "vec3": [valid_vector5, valid_vector5, valid_vector5, valid_vector5, valid_vector4],
    "scalar1": [1, 2, 3, np.nan, 5],
    "scalar2": [10, 20, 30, 40, 50]
}

df_test = pd.DataFrame(data)
print("Original DataFrame:")
print(df_test)

# Define vector_cols as a list of lists:
# - The first group consists of 'vec1' and 'vec2' which must each be of length 4.
# - The second group consists of 'vec3' which must be of length 5.
vector_cols = [["vec1", "vec2"], ["vec3"]]
scalar_cols = ["scalar1", "scalar2"]

# Filter the DataFrame.
filtered_df = valid_df_rows_iter(df_test, vector_cols, lengths=[4, 5])
print("\nFiltered DataFrame:")
filtered_df


Original DataFrame:
                   vec1                  vec2                       vec3  \
0  [0.0, 1.0, 2.0, 3.0]  [0.0, 1.0, 2.0, 3.0]  [0.0, 1.0, 2.0, 3.0, 4.0]   
1  [0.0, 1.0, 2.0, 3.0]  [0.0, 1.0, nan, 3.0]  [0.0, 1.0, 2.0, 3.0, 4.0]   
2            [0.0, 1.0]  [0.0, 1.0, 2.0, 3.0]  [0.0, 1.0, 2.0, 3.0, 4.0]   
3  [0.0, 1.0, 2.0, 3.0]  [0.0, 1.0, 2.0, 3.0]  [0.0, 1.0, 2.0, 3.0, 4.0]   
4  [0.0, 1.0, nan, 3.0]  [0.0, 1.0, 2.0, 3.0]       [0.0, 1.0, 2.0, 3.0]   

   scalar1  scalar2  
0      1.0       10  
1      2.0       20  
2      3.0       30  
3      NaN       40  
4      5.0       50  

Filtered DataFrame:


,vec1,vec2,vec3,scalar1,scalar2
0,"[0.0, 1.0, 2.0, 3.0]","[0.0, 1.0, 2.0, 3.0]","[0.0, 1.0, 2.0, 3.0, 4.0]",1.0,10
